# Inference & Usage — Tüm Detaylar

Bu notebook **smoke test ile kanıtlanmış** çalışır (`smoke/08_inference_smoke.py`).

## Hedef
Eğitim sonrası (veya öncesi) bir modeli **doğru kullanmayı** öğretmek:
- Model yükleme yöntemleri (Unsloth Fast vs vanilla HF)
- Chat template + `add_generation_prompt`
- Generation parametreleri (model-bazlı önerilenler)
- Streaming (basit ve production)
- Batched inference
- LoRA kaydet / yeniden yükle
- Save formatları (LoRA / merged / GGUF)

## Test Edildi (RTX 4070 Ti SUPER 16GB)
Tüm 9 pattern smoke test'te geçti — `Qwen3-4B-Instruct-2507` 4-bit + LoRA r=16.

## İçindekiler

1. **Model Yükleme** — Unsloth FastLanguageModel
2. **`for_inference()` Optimizasyonu** — 2x daha hızlı
3. **Chat Template + `apply_chat_template`**
4. **Generation Parametreleri** — temperature, top_p, top_k, min_p, repetition_penalty
5. **Model-Bazlı Önerilen Değerler**
6. **Streaming** — TextStreamer vs TextIteratorStreamer
7. **Batched Inference**
8. **LoRA Kaydet ve Yeniden Yükle**
9. **Save Formatları** — LoRA / Merged / GGUF
10. **Thinking Model Inference** — `<think>` parse ve kontrol
11. **Tool Calling Inference** — Multi-turn tool execution
12. **Yaygın Tuzaklar**

## 1. Model Yükleme

İki ana path:

### A. Unsloth (önerilir — eğitim de yapacaksanız)
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen3-4B-Instruct-2507',
    max_seq_length=2048, load_in_4bit=True,
)
```

### B. Vanilla HuggingFace (sadece inference)
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.bfloat16, device_map='auto',
)
```

**Hangisi?**
- Eğitim de yapacaksan → Unsloth (Triton kernels, daha hızlı + az VRAM)
- Sadece inference + production → Vanilla daha esnek (vLLM, TGI gibi serving framework'leri ile)

In [ ]:
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
from transformers import TextStreamer, TextIteratorStreamer
from threading import Thread

MODEL = 'unsloth/Qwen3-4B-Instruct-2507'

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL,
    max_seq_length = 2048,
    load_in_4bit = True,                       # QLoRA — 4 GB peak
    full_finetuning = False,
)

# Chat template — doğru format için
tokenizer = get_chat_template(tokenizer, chat_template='qwen3-instruct')

## 2. `for_inference()` Optimizasyonu

**KRİTİK:** Inference yaparken `FastLanguageModel.for_inference(model)` çağır. **2x daha hızlı**.

Training yapacaksan: `FastLanguageModel.for_training(model)` ile geri dön.

Ne yapıyor: Cache'leri optimize ediyor, gradient hesaplamayı kapatıyor, dropout'ları kaldırıyor.

In [ ]:
FastLanguageModel.for_inference(model)            # 2x faster

## 3. Chat Template Inference Kullanımı

**`add_generation_prompt=True`** zorunlu — model'e "şimdi cevap ver" işareti ekler.

### Pattern
```python
messages = [
    {'role': 'system', 'content': '...'},      # opsiyonel
    {'role': 'user', 'content': '...'},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,                  # ZORUNLU
)
inputs = tokenizer(text, return_tensors='pt').to('cuda')
out = model.generate(**inputs, ...)
```

In [ ]:
# Helper — chat template + generate + decode-only-new-tokens
def gen_text(messages, **gen_kwargs):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, **gen_kwargs)
    new_tokens = out[0][inputs['input_ids'].shape[1]:]   # KRITIK: prompt'u skip
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

msgs = [{'role': 'user', 'content': 'Faiz nedir?'}]
print(gen_text(msgs, max_new_tokens=80, do_sample=False))   # greedy

## 4. Generation Parametreleri

### Sampling Parametreleri
| Parametre | Anlamı | Tipik Değer |
|---|---|---|
| `temperature` | Yaratıcılık. Düşük = deterministic, yüksek = çeşitli | 0.7-1.0 |
| `top_p` (nucleus) | Cumulative prob threshold — bu eşiği geçen token'lardan seç | 0.8-0.95 |
| `top_k` | İlk K token'dan seç | 20-64 |
| `min_p` | Minimum probability threshold (top_p alternatifi) | 0.05-0.1 |
| `do_sample` | False = greedy/argmax, True = sampling | True |

### Tekrar Kontrolü
| Parametre | Anlamı | Tipik Değer |
|---|---|---|
| `repetition_penalty` | Tekrar eden token'lara ceza (>1) | 1.0-1.3 |
| `no_repeat_ngram_size` | Bu boyutta n-gram tekrar etmesin | 3 (uzun text için) |

### Length Kontrolü
| Parametre | Anlamı |
|---|---|
| `max_new_tokens` | **Sadece yeni** token sayısı (önerilir) |
| `max_length` | Toplam (prompt + new) — kullanma! |
| `min_new_tokens` | En az kaç yeni token |

In [ ]:
# Greedy (deterministic)
print('--- Greedy ---')
print(gen_text(msgs, max_new_tokens=60, do_sample=False))

# Düşük temperature (focused)
print('\n--- T=0.3 (focused) ---')
print(gen_text(msgs, max_new_tokens=60, temperature=0.3, do_sample=True))

# Yüksek temperature (creative)
print('\n--- T=1.5 (creative) ---')
print(gen_text(msgs, max_new_tokens=60, temperature=1.5, top_p=0.95, do_sample=True))

# Tekrar cezası
print('\n--- repetition_penalty=1.3 ---')
print(gen_text(msgs, max_new_tokens=60,
               temperature=0.7, top_p=0.8, top_k=20,
               repetition_penalty=1.3, do_sample=True))

## 5. Model-Bazlı Önerilen Değerler

Resmi notebook'lardan derlenmiş:

| Model Ailesi | Temperature | top_p | top_k | min_p | Notlar |
|---|---|---|---|---|---|
| **Qwen3 Instruct (non-thinking)** | 0.7 | 0.8 | 20 | – | Daha düz, focused |
| **Qwen3 Thinking** | 0.6 | 0.95 | 20 | – | Düşük T thinking için |
| **Qwen3.5 Vision** | 1.5 | – | – | 0.1 | min_p kullanır |
| **Gemma 4** | 1.0 | 0.95 | 64 | – | Daha esnek |
| **Gemma 3** | 1.0 | 0.95 | 64 | – | Aynı Gemma 4 |
| **Llama 3.1** | 0.6 | 0.9 | – | – | Düşük T |

### Use-case Bazlı
| Use Case | T | top_p | Notlar |
|---|---|---|---|
| Code generation | 0.0-0.2 | 1.0 | Greedy/low T daha güvenli |
| Doğrulanabilir cevap (math, fact) | 0.0-0.3 | 1.0 | Deterministic |
| Yaratıcı yazı | 0.8-1.2 | 0.95 | Daha çeşitli |
| Brainstorming | 1.2-1.5 | 0.95 | Yüksek çeşitlilik |

## 6. Streaming

İki seçenek:

### A. `TextStreamer` (basit)
Sadece stdout'a yazar. Notebook'ta görünür.

### B. `TextIteratorStreamer` (production)
Async-friendly, FastAPI / Streamlit / Discord bot için ideal. Background thread'de generate çalışır, ana thread'de yield.

In [ ]:
# A. TextStreamer — basit
text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to('cuda')
_ = model.generate(
    **inputs, max_new_tokens=80,
    temperature=0.7, top_p=0.8, top_k=20, do_sample=True,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

In [ ]:
# B. TextIteratorStreamer — production pattern
streamer = TextIteratorStreamer(
    tokenizer, skip_prompt=True, skip_special_tokens=True,
)
gen_kwargs = dict(
    **inputs, max_new_tokens=80,
    temperature=0.7, top_p=0.8, top_k=20, do_sample=True,
    streamer=streamer,
)
thread = Thread(target=model.generate, kwargs=gen_kwargs)
thread.start()

# Token-by-token yield
collected = ''
for chunk in streamer:
    collected += chunk
    print(chunk, end='', flush=True)
thread.join()
print(f'\n\nTotal: {len(collected)} chars')

## 7. Batched Inference

Birden çok prompt aynı anda. **Padding LEFT** zorunlu.

Memory: batch_size * (prompt_len + max_new_tokens) * model_size = O(batch * seq * params)

In [ ]:
prompts = [
    'Istanbul nerededir?',
    'Kar nasil olusur?',
    'Quantum nedir?',
]
formatted = [
    tokenizer.apply_chat_template(
        [{'role': 'user', 'content': p}],
        tokenize=False, add_generation_prompt=True,
    ) for p in prompts
]

tokenizer.padding_side = 'left'                  # ZORUNLU batch icin
batch_inputs = tokenizer(
    formatted, return_tensors='pt', padding=True,
).to('cuda')

out = model.generate(
    **batch_inputs, max_new_tokens=50,
    temperature=0.7, top_p=0.8, top_k=20, do_sample=True,
    pad_token_id=tokenizer.pad_token_id,
)

# Decode each sample
for i, prompt in enumerate(prompts):
    new_tokens = out[i][batch_inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    print(f'\nQ: {prompt}')
    print(f'A: {response[:120]}...')

tokenizer.padding_side = 'right'                 # restore

## 8. LoRA Kaydet ve Yeniden Yükle

### Kaydet (eğitim sonrası)
```python
model.save_pretrained('my_lora')
tokenizer.save_pretrained('my_lora')
```
Bu sadece **adapter weights** + config kaydeder (~150MB), base model değil.

### Yeniden Yükle (production scenario)
```python
# from_pretrained ile direkt LoRA dir'ini ver
# Base model adapter_config.json'dan otomatik bulunur
model, tokenizer = FastLanguageModel.from_pretrained(
    'my_lora',                                    # LoRA dir, base ADI değil
    max_seq_length=2048, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```

Unsloth LoRA dir'inden `adapter_config.json`'ı okuyup base model'i otomatik indirir + yükler + adapter'ı bağlar.

## 9. Save Formatları

### A. LoRA Adapter (en küçük, ~150MB)
```python
model.save_pretrained('my_lora')
tokenizer.save_pretrained('my_lora')
```
**Avantaj:** Küçük, hızlı upload/download. Birden çok adapter aynı base ile kullanılabilir.
**Dezavantaj:** Inference için her zaman base model gerekiyor.

### B. Merged 16-bit (vLLM / HF inference)
```python
model.save_pretrained_merged(
    'my_merged',
    tokenizer,
    save_method='merged_16bit',                  # 'merged_4bit' de var
)
```
**Avantaj:** vLLM, TGI gibi production serving framework'lerinde direkt kullanılır. Tek dosya.
**Dezavantaj:** Tam model boyutu (Qwen3-4B = ~8GB bf16).

### C. GGUF (Ollama / llama.cpp)
```python
model.save_pretrained_gguf(
    'my_gguf',
    tokenizer,
    quantization_method='q4_k_m',                # q4_k_m, q5_k_m, q8_0, f16
)
```
**Avantaj:** CPU + GPU hybrid (Ollama, llama.cpp). Quantize edilmiş, küçük.
**Dezavantaj:** llama.cpp bağımlılığı; conversion zaman alır.

### D. Hub'a Push
```python
# LoRA
model.push_to_hub('USER/my_lora', token='hf_xxx')
tokenizer.push_to_hub('USER/my_lora', token='hf_xxx')

# Merged
model.push_to_hub_merged(
    'USER/my_merged', tokenizer,
    save_method='merged_16bit', token='hf_xxx',
)
```

### Karar Tablosu
| Hedef | Format | Sebep |
|---|---|---|
| Hub'a deploy / paylaş | LoRA | Küçük, çok adapter aynı base |
| vLLM serving | merged_16bit | vLLM LoRA da destekler ama merged daha basit |
| Ollama / local CLI | GGUF q4_k_m | CPU+GPU, quantized |
| Mobile / edge | GGUF q4_k_s | En küçük |
| HuggingFace Inference API | merged_16bit | Native HF format |

## 10. Thinking Model Inference

Thinking model'ler (Qwen3-Thinking-2507, R1-distill, vs.) yanıtı **iki bölüm** halinde üretir:
1. `<think>...</think>` — model'in iç muhakemesi (UI'de gizlenebilir)
2. **Final answer** — temiz, kullanıcıya yansıyan kısım

### Generation Akışı
```
User: 137 * 49 nedir?
        ↓ (qwen3-thinking template generation prompt sonu: <|im_start|>assistant\n<think>\n)
Model: 137 * 50 = 6850, 6850 - 137 = 6713.</think>

Sonuç 6713.<|im_end|>
```

### Kontrol Mekanizmaları
| Yöntem | Sonuç |
|---|---|
| `qwen3-thinking` template | Auto `<think>\n` ekler (her zaman) |
| `qwen3-instruct` template | `<think>` eklemez (thinking yok) |
| `enable_thinking=False` parametresi | Bazı modellerde devre dışı (Gemma 4 destekliyor, qwen3-thinking ignore eder) |
| Post-process strip | Regex ile `<think>...</think>` kaldır → final answer çek |

### Önerilen Generation Parametreleri
| Param | Değer |
|---|---|
| `temperature` | **0.6** (düşük — odaklı reasoning) |
| `top_p` | 0.95 |
| `top_k` | 20 |
| `max_new_tokens` | **1024-4096** (think + answer uzun!) |
| `do_sample` | True |

In [ ]:
# Thinking model yükle (önceki sectionlarda Qwen3-Instruct yüklemiştik)
# Yeni session simülasyonu için modeli temizle
import gc
del model
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen3-4B-Thinking-2507',
    max_seq_length = 4096,                       # think uzun olabilir
    load_in_4bit = True,
)
tokenizer = get_chat_template(tokenizer, chat_template='qwen3-thinking')
FastLanguageModel.for_inference(model)

In [ ]:
# Math problem — thinking model için ideal
msgs = [{'role': 'user', 'content': 'Hesapla 137 * 49'}]
text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to('cuda')

out = model.generate(
    **inputs, max_new_tokens=1024,                # think için yer aç
    temperature=0.6, top_p=0.95, top_k=20,        # thinking modeli önerisi
    do_sample=True,
)
full = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'--- Full output ({len(full)} chars) ---')
print(full[:800])

### Parse — `<think>` ve Final Answer Ayır

**Önemli quirk (smoke test'te kanıtlandı):** `qwen3-thinking` template'i generation prompt'unun sonuna `<think>\n` ekler. Decode `skip_special_tokens=True` ile **bu açılış token'ı kayıp olur** — ama `</think>` capture edilir.

Pattern: önce full `<think>...</think>` regex dene, başarısız olursa `</think>` ile split.

In [ ]:
import re

# Strategy 1: <think>...</think> tam yakalama
think_match = re.search(r'<think>\s*(.*?)\s*</think>', full, re.DOTALL)
if think_match:
    think_part = think_match.group(1).strip()
    final_part = re.sub(r'<think>.*?</think>\s*', '', full, count=1, flags=re.DOTALL).strip()
else:
    # Strategy 2: </think>'a kadar split (template prepended <think>, decoder dropped it)
    if '</think>' in full:
        think_part, final_part = full.split('</think>', 1)
        think_part = think_part.strip()
        final_part = final_part.strip()
    else:
        think_part = ''
        final_part = full.strip()

print(f'--- Thinking ({len(think_part)} chars) ---')
print(think_part[:400] + ('...' if len(think_part) > 400 else ''))
print(f'\n--- Final Answer ({len(final_part)} chars) ---')
print(final_part[:400])

print(f'\n--- For UI (only final to user): ---')
print(final_part)

### Production UI Pattern — Streaming + Thinking

```python
in_think = True
think_buffer = ''
answer_buffer = ''

for chunk in streamer:
    if '</think>' in chunk:
        # transition: think → answer
        before, after = chunk.split('</think>', 1)
        think_buffer += before
        answer_buffer += after
        in_think = False
        ui.show_thinking_collapsed(think_buffer)   # 'Thinking...' badge
    elif in_think:
        think_buffer += chunk
        ui.update_thinking_indicator()              # spinner
    else:
        answer_buffer += chunk
        ui.append_to_answer(chunk)                  # ana yanıt
```

Bu pattern Gemini, Claude, ChatGPT'nin reasoning model UI'sinde kullanılan yaklaşım.

## 11. Tool Calling Inference

Tool calling = model'in **fonksiyon çağırma** yetkisi. Model JSON formatında tool çağırır, kodun execute edip sonucu geri verir.

### 3-Aşamalı Akış (smoke test'te kanıtlandı)
```
[User]   → "Istanbul hava nasıl?"
             ↓ (system'e tools schema enjekte edilir)
[Model]  → <tool_call>{"name":"get_weather","arguments":{"city":"Istanbul"}}</tool_call>
             ↓ (kodun çağırıyor)
[Tool]   → {"city":"Istanbul","temp_c":18,"condition":"cloudy"}
             ↓ (model'e geri ver, role='tool')
[Model]  → "İstanbul'un hava durumu şu anda bulutlu ve sıcaklığı 18°C'ye ulaşmaktadır."
```

### Format Modeli Bazlı (Parse Regex)
| Model | Format | Parse Regex |
|---|---|---|
| Qwen3 Instruct/Thinking | `<tool_call>{json}</tool_call>` | `r'<tool_call>\s*(\{.*?\})\s*</tool_call>'` |
| Qwen3.5 (Hermes-Pro) | `<function=NAME><parameter=KEY>VAL</parameter></function>` | `r'<function=(\w+)>(.*?)</function>'` |
| Gemma 4 (FunctionGemma) | `<\|tool_call>call:NAME{...}<tool_call\|>` | `r'<\|tool_call>call:(\w+)\{(.*?)\}<tool_call\|>'` |
| Llama 3.1 | Plain JSON in content | Direct json.loads |

### Önerilen Generation Parametreleri
| Param | Değer |
|---|---|
| `temperature` | **0.3-0.7** (düşük — JSON format'ı bozulmasın) |
| `top_p` | 0.8 |
| `top_k` | 20 |
| `max_new_tokens` | 200-300 (tool call kısa, final cevap orta) |

In [ ]:
# Qwen3-Instruct yükle (tool calling için ideal)
import gc
del model
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen3-4B-Instruct-2507',
    max_seq_length = 2048,
    load_in_4bit = True,
)
tokenizer = get_chat_template(tokenizer, chat_template='qwen3-instruct')
FastLanguageModel.for_inference(model)

In [ ]:
# Tools tanımla — OpenAI function-calling JSON schema
import json

tools = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get current weather for a city',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string', 'description': 'City name'}},
            'required': ['city'],
        },
    },
}, {
    'type': 'function',
    'function': {
        'name': 'calculator',
        'description': 'Evaluate a math expression',
        'parameters': {
            'type': 'object',
            'properties': {'expression': {'type': 'string'}},
            'required': ['expression'],
        },
    },
}]

# Mock tool implementations
def get_weather(city: str) -> dict:
    fake = {'Istanbul': 18, 'Ankara': 12, 'Paris': 15, 'Tokyo': 22}
    return {'city': city, 'temp_c': fake.get(city, 20), 'condition': 'cloudy'}

def calculator(expression: str) -> str:
    try:
        return str(eval(expression, {'__builtins__': {}}, {}))
    except Exception as e:
        return f'Error: {e}'

TOOL_FNS = {'get_weather': get_weather, 'calculator': calculator}

In [ ]:
# Turn 1 — User asks, model decides to call tool
conv = [
    {'role': 'system', 'content': 'You are a helpful assistant with tool access.'},
    {'role': 'user',   'content': 'Istanbul hava nasil?'},
]

text = tokenizer.apply_chat_template(
    conv, tokenize=False, tools=tools, add_generation_prompt=True,
)
inputs = tokenizer(text, return_tensors='pt').to('cuda')
out = model.generate(
    **inputs, max_new_tokens=200,
    temperature=0.3, top_p=0.8, top_k=20, do_sample=True,    # düşük T — JSON precision
)
response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('--- Model output ---')
print(response)

In [ ]:
# Parse <tool_call>{json}</tool_call> (Qwen3 format) + execute
tool_match = re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', response, re.DOTALL)

if tool_match:
    call = json.loads(tool_match.group(1))
    tool_name = call['name']
    tool_args = call['arguments']
    print(f'Parsed call: {tool_name}({tool_args})')

    # Execute (real Python function)
    if tool_name in TOOL_FNS:
        result = TOOL_FNS[tool_name](**tool_args)
        print(f'Executed → {result}')
    else:
        result = {'error': f'Unknown tool: {tool_name}'}
else:
    print('No tool call detected — model answered directly')
    tool_name = None
    result = None

In [ ]:
# Turn 2 — Feed tool result, get final answer
if tool_name:
    conv_full = conv + [
        # Asistan'ın tool çağrısı
        {'role': 'assistant', 'content': '',
         'tool_calls': [{
             'id': 'call_1', 'type': 'function',
             'function': {'name': tool_name, 'arguments': tool_args},
         }]},
        # Tool execution sonucu (role='tool', tool_call_id eşleşmeli)
        {'role': 'tool', 'tool_call_id': 'call_1', 'name': tool_name,
         'content': json.dumps(result)},
    ]

    text2 = tokenizer.apply_chat_template(
        conv_full, tokenize=False, tools=tools, add_generation_prompt=True,
    )
    inputs2 = tokenizer(text2, return_tensors='pt').to('cuda')
    out2 = model.generate(
        **inputs2, max_new_tokens=150,
        temperature=0.7, top_p=0.8, top_k=20, do_sample=True,
    )
    final = tokenizer.decode(out2[0][inputs2['input_ids'].shape[1]:], skip_special_tokens=True)
    print('--- Final answer ---')
    print(final)

### Production Tool Loop (Multi-step)

Model birden fazla tool çağırabilir (parallel ya da sequential). Production pattern:

```python
MAX_ITERATIONS = 5                                # sonsuz döngü engeli
conv = [{'role': 'user', 'content': user_query}]

for i in range(MAX_ITERATIONS):
    # 1. Generate
    response = generate(conv, tools=tools)

    # 2. Parse all tool calls
    calls = parse_all_tool_calls(response)

    # 3. No tool calls → final answer, done
    if not calls:
        conv.append({'role': 'assistant', 'content': response})
        break

    # 4. Append assistant message with tool_calls
    conv.append({'role': 'assistant', 'content': '',
                 'tool_calls': [...]})

    # 5. Execute each tool, append results
    for call in calls:
        result = TOOL_FNS[call['name']](**call['args'])
        conv.append({'role': 'tool', 'tool_call_id': call['id'],
                     'name': call['name'], 'content': json.dumps(result)})

# Loop ends when model gives final answer (no more tool calls)
```

### Kontrol Tuzakları
1. **Tools desteklemeyen template'te `tools=[...]`** → sessiz ignore (Gemma 4 standart)
2. **`tool_call_id` eşleşmiyor** → model takılır
3. **JSON format'ı yanlış** → tool execute edilmez (validate et: `json.loads` try/except)
4. **Sonsuz tool call döngüsü** → `MAX_ITERATIONS` koy
5. **Tool result çok uzun** → context'i şişirir, kısalt veya summarize et
6. **`temperature` yüksek** → JSON format'ı bozulur (düşük tut: 0.3-0.7)

## 12. Yaygın Tuzaklar

| # | Hata | Sonuç | Çözüm |
|---|---|---|---|
| 1 | `add_generation_prompt=False` inference'ta | Model cevap vermez, training format'ında bekler | `=True` |
| 2 | `for_inference()` çağırmamak | 2x yavaş inference | `FastLanguageModel.for_inference(model)` |
| 3 | Decoded text'te prompt da var | Output'a prompt karışır | `out[0][inputs['input_ids'].shape[1]:]` |
| 4 | Batch'te `padding_side='right'` | Yanlış output (pad token'lar response'un başına gider) | `'left'` zorunlu |
| 5 | `do_sample=False` + `temperature` | Temperature ignore edilir | `do_sample=True` koy |
| 6 | `max_length` kullanmak | Prompt + completion toplam, kontrol zor | `max_new_tokens` |
| 7 | `model.generate(input_ids=..., temperature=...)` (kwargs unpacking) | Bazı durumlarda hata | `**inputs` ile geç |
| 8 | Vision modelinde plain text content | TypeError | `[{'type': 'text', 'text': ...}]` list |
| 9 | `tokenizer.decode(out)` (yeniden whole sequence) | Output'a prompt + special tokens dahil | `skip_special_tokens=True` + slice |
| 10 | Production'da `TextStreamer` | Async olmaz (stdout'a yazar) | `TextIteratorStreamer` |
| 11 | Reload'da base model adı yazmak | `from_pretrained('Qwen3-4B')` adapter'ı atlar | `from_pretrained('my_lora_dir/')` |
| 12 | Generation parametrelerini sürekli aynı tutmak | Use-case fark eder | Code → low T, creative → high T |
| 13 | Thinking model `<think>` tam regex'le yakalanmıyor | Template prepended `<think>\n`, decoder skip etti | `</think>` ile split fallback |
| 14 | Tool calling `temperature` yüksek | JSON format bozulur | 0.3-0.7 düşük tut |
| 15 | Tool result çok uzun context'e dahil | Loop'lar yavaşlar, OOM | Tool result'ı kısalt/summarize |

## Özet

1. **`FastLanguageModel.for_inference(model)`** — 2x hız
2. **`add_generation_prompt=True`** inference'ta zorunlu
3. **`out[0][inputs['input_ids'].shape[1]:]`** — sadece yeni token'ları decode et
4. **Generation params model-bazlı** — Qwen3 instruct T=0.7, Gemma 4 T=1.0, Qwen3.5 min_p=0.1
5. **Batch'te `padding_side='left'`** zorunlu
6. **`TextIteratorStreamer`** production'da (FastAPI / Streamlit)
7. **LoRA reload:** `from_pretrained('lora_dir/')` — adapter_config'ten base'i otomatik bulur
8. **Save: LoRA (paylaş)** / **Merged (vLLM)** / **GGUF (Ollama)**

**Test edildi:** `smoke/08_inference_smoke.py` — 9 farklı pattern (load, params, streaming, batch, train, save, reload) hepsi geçti.

## Sonraki adımlar
- **Production serving:** vLLM, TGI, Ollama dökümentasyonları
- **Adapter merging:** Birden çok LoRA adapter'ı birleştirme (`add_weighted_adapter`)
- **Quantization:** AWQ, GPTQ, AQLM (post-training quantization)